<a href="https://colab.research.google.com/github/LucasAbCV/procesamientoDelHabla/blob/main/TP3_LucasBruzzo_chatbot_retrieval_template_parte_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP3 Chatbots basados en recuperación de la información

En inglés information retrieval chatbots

# Motor de búsqueda

* Búsqueda por palabras clave: Extrae palabras clave de la pregunta del usuario y busca coincidencias en las preguntas almacenadas.

* Similitud del coseno: Si has representado las preguntas como vectores (por ejemplo, usando TF-IDF o word embeddings), puedes usar la similitud del coseno para medir la distancia entre las preguntas.

* Embeddings: Utiliza modelos de word embeddings como por ejemplo Word2Vec para obtener representaciones semánticas de las preguntas y las consultas del usuario.

## Librerías

Debes trabajar en Python. Puedes usar las librerías sklearn, pandas, spacy o nltk o gensim para el punto de usar o buscar embeddings.

In [ ]:
#Instalaciones
!pip install spacy --quiet                                                          #Librería spaCy para procesamiento de lenguaje.
#!python -m spacy download es_core_news_sm --quiet                                   #Modelo pequeño en español (Tokenización, lematización y stopwords).
! pip install gensim matplotlib numpy scikit-learn                                  #Para embeddings, gráficos, arrays y cálculos de vectores, y scikit-learn.
!python -m spacy download es_core_news_md                                           #Modelo en español mediano.


#Fuente: la parte de spaCY se extrajo del notebook de la semana 5 sobre Embeddings. Las librerías instaladas se agregaron a medida que se completaron las demàs celdas. El modelo en español de tamaño mediano fue agregado en base al
#notebook 3_1_spacy_embeddings_ejemplos.ipynb de la semana 5

In [ ]:
#Librerías
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import unicodedata
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from numpy import dot
from numpy.linalg import norm
import gensim.downloader as api
import spacy
import pandas as pd


#Fuente:se fue completando a medida que se fueron escribiendo las demàs celdas

## Actividades

#1) Elaborar un dataset de preguntas y respuestas para crear un Chatbot para un aplicación particular. ( 3 puntos )

###Debe definir la aplicación (atención al cliente bancario, atención a estudiantes universitarios, etc).

La aplicación del chatbot será atención al cliente en una biblioteca

###El listado de preguntas y respuestas debe tener como mínimo 20 elementos pregunta - respuesta.

In [ ]:
#Diccionaro con preguntas-respuestas
pregresp = {
    "Hola": "hola buen día.",
    "¿A qué hora cierra la biblioteca?": "La biblioteca cierra a las 20hs.",
    "¿A qué hora abre la biblioteca?": "La biblioteca abre a las 9hs.",
    "Adios": "Adios, hasta luego.",
    "¿Qué días abre la biblioteca?": "La biblioteca abre todos los días de la semana.",
    "¿Dónde está la biblioteca?": "La biblioteca está en la calle 123.",
    "¿Cuál es el número de teléfono de la biblioteca?": "El número de teléfono de la biblioteca es 123456789.",
    "¿Tienen libros de ciencia ficción?": "Si, la biblioteca tiene libros de ciencia ficción.",
    "¿Tienen libros de historia?": "Si, la biblioteca tiene libros de historia.",
    "¿Tienen libros de programación?": "Si, la biblioteca tiene libros de programación.",
    "¿Tienen libros de química?": "Si, la biblioteca tiene libros de química.",
    "¿Tienen libros de física?": "Si, la biblioteca tiene libros de física.",
    "¿Tienen libros de matemáticas?": "Si, la biblioteca tiene libros de matemáticas.",
    "¿Tienen libros de biología?": "Si, la biblioteca tiene libros de biología.",
    "¿Tienen libros de geografía?": "Si, la biblioteca tiene libros de geografía.",
    "¿Tienen libros en inglés?": "No, la biblioteca no tiene libros en inglés.",
    "¿Tienen libros de francés?": "No, la biblioteca no tiene libros de francés.",
    "¿Tienen libros de alemán?": "No, la biblioteca no tiene libros de alemán.",
    "¿Tienen libros de italiano?": "No, la biblioteca no tiene libros de italiano.",
    "¿Tienen libros de autoayuda?": "No, la biblioteca no tiene libros de autoayuda.",
    "Gracias": "De nada",
    "¿Abren los feriados?": "No, la biblioteca permanece cerrada los feriados.",
    "¿Tiene estacionamiento?": "Sí, hay un estacionamiento gratuito para visitantes.",
    "¿Tienen correo electrónico?": "Sí, podés escribir a biblioteca@ejemplo.com.",
    "Quiero hablar con una persona": "Te derivo con el personal del mostrador, llamá al 123456789.",
    "¿Tienen wifi?": "Sí, el wifi es gratuito y la clave está en el mostrador.",
    "¿Se puede estudiar en la biblioteca?": "Sí, hay salas de lectura y de estudio silenciosas.",
    "¿Tienen computadoras para usar?": "Sí, hay computadoras disponibles por hasta 1 hora por persona.",
    "¿Cómo me hago socio?": "Presentá tu DNI en el mostrador y completá el formulario de inscripción.",
    "¿Cuánto cuesta ser socio?": "La inscripción es gratuita.",
    "¿Cuántos libros puedo llevar prestados?": "Podés llevar hasta 3 libros a la vez.",
    "¿Por cuánto tiempo puedo llevarme un libro?": "El préstamo dura 15 días.",
    "¿Puedo renovar un préstamo?": "Sí, podés renovarlo una vez por 15 días más si nadie lo reservó.",
    "¿Qué pasa si devuelvo un libro tarde?": "Se aplica una suspensión de 1 día por cada día de demora.",
    "¿Qué hago si perdí un libro?": "Acercate al mostrador para reponerlo o abonar su valor.",
    "¿Puedo reservar un libro?": "Sí, podés reservarlo en el mostrador o por teléfono.",
}





#2) Crear el chatbot utilizando TFIDF y similitud del coseno. (1 punto)



###La idea es crear la mayor cantidad de funciones que sirvan tanto para este chatbot como el de la actividad siguiente.

In [ ]:
#Esta función sirve para limpiar texto, y se aplica al texto utilizado de ambos modelos
def limpiar(texto):
    texto = texto.lower()                                      #Pasaje a miníscula
    # quitar todo lo que no sea letra, número o espacio
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)                 #Reemplaza con un espacio todo lo que no sea una letra minúscula, número o espacio
    texto = re.sub(r"\s+", " ", texto).strip()                 #Ordena los espacios
    return texto

#Fuente: se combinó lo visto en limpieza de textos en clases anteriores e información de https://docs.python.org/3/library/re.html#re.sub


In [ ]:
#Se separan las preguntas de las respuestas
preguntas = list(pregresp.keys())                              #Preguntas
respuestas = list(pregresp.values())                           #Respuestas
preguntas_limpias = [limpiar(p) for p in preguntas]            #Limpia cada pregunta


#Fuente: aplicación de teoría en diccionarios para obtener cada parte por separado e iteración sobre listas para limpiar cada pregunta.

###TFIDF

In [ ]:
vectorizer = TfidfVectorizer()                          #Prepoaración del vectorizador
matriz = vectorizer.fit_transform(preguntas_limpias)   #Lee las preguntas, arma le vocabulario calculando que tan informativa es cada una. También convierte cada pregunta en un vector numérico según el vocabulario (incluye la tokenización)

#Fuente:notebook de la semana 4 llamado Chatbot-retrieval-.ipynb


# 3) Crear otro chatbot utilizando embeddings. Indique cuál embedding (1 punto) pre-entrenado de las librerías Spacy o nltk o gensim eligió. Justifique.

###Embeddings

In [ ]:
nlp = spacy.load("es_core_news_md")                                                    #Modelo basado
#Fuente: notebook 3_1_spacy_embeddings_ejemplos.ipynb de la clase 5

Se eligió spaCy porque ofrece modelos oficiales en español (es_core_news_md) que incluyen embeddings preentrenados, de tokenización, stopwords y detección de puntuación del idioma, facilitanddo el preprocesamiento y la vectorización de las consultas.

In [ ]:
def vectorizar_texto(texto_limpio):

    doc = nlp(texto_limpio)                                                 #Procesa el texto con el modelo spaCy y devuelve un Doc que es el texto tokenizado con información para cada token.


    #Lista que conserva los vectores si no son stopwords, signos de puntuación y palabras ddesconocidas para el modelo
    vectores = [
        token.vector
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and token.has_vector
    ]
    #Caso de que la lista este vacía, devuelve un vector de ceros
    if not vectores:
        return np.zeros(nlp.vocab.vectors.shape[1])

    embedding = np.mean(vectores, axis=0)                                       #Para cada dimensión del vector, se promedia el valor de las filas.
    return embedding



#Fuente: https://chatgpt.com/share/6b0118a4-7e98-83e9-b6af-f971066bccde

In [ ]:
matriz_emb = np.array([vectorizar_texto(p) for p in preguntas_limpias])                            #Base de conocimiento vectorizada (pila de vectores utilizando embeddings)


#Fuente: se utilizó conocimiento aprendido bajo la lógica de aplicar la función antes solicitada.

In [ ]:
def responder(consulta, metodo="emb"):
    consulta_limpia = limpiar(consulta)                                                                       #Limpia el texto que entra como consulta

    #Para que responda el chatbot TFIDF (cuenta palabras y pondera por rareza)
    if metodo == "tfidf":
        vec = vectorizer.transform([consulta_limpia])                                                         #Convierte un texto a un vector numérico
        sims = cosine_similarity(vec, matriz).flatten()                                                       #Similitud del coseno: semejanza de la dirección de 2 vectores. Se toma el vector anterior y se lo compara con todos los vectores de matriz
        umbral = 0.2                                                                                          #El umbral sirve para evaluar que tan bueno es el vector mas similar, si es suficientemente aceptable
    #Para que responda el chatbot con embedding
    else:
        #El proceso es similar para embeddings solo que cambia la vectorización y aca se capturan significados
        vec = vectorizar_texto(consulta_limpia).reshape(1, -1)
        sims = cosine_similarity(vec, matriz_emb).flatten()
        umbral = 0.70

    mejor = sims.argmax()                                                                                     #Obtiene el índice del vector más similar
    if sims[mejor] < umbral:
        return "Perdón, no se entendio la pregunta"                                                           #Si el vector mas similar no es aceptable se imprime una respuesta pre-establecida
    return respuestas[mejor]                                                                                  #Si el vector mas similar es aceptable se imprime la respuesta correspondiente


#Fuente: se adaptó la función "responder_pregunta" del notebook de la semana 4 llamado Chatbot-retrieval-.ipynb


#4) Muestra ambos chatbots funcionando (1 punto)



##Adjuntar la lista de preguntas y respuestas utilizadas para probar el funcionamiento.



In [ ]:

pregresp_test = {

    # ==========================================
    # 1. PARAFRASEO - HORARIOS
    # ==========================================

    "¿Hasta qué hora se puede entrar a la biblioteca?":
        "La biblioteca cierra a las 20hs.",

    "¿A partir de qué hora está abierta la biblioteca al público?":
        "La biblioteca abre a las 9hs.",

    "Si quisiera ir a la biblioteca temprano, ¿a qué hora tendría que estar allí?":
        "La biblioteca abre a las 9hs.",

    "¿A qué hora dejan de atender?":
        "La biblioteca cierra a las 20hs.",

    "¿Cuál es el horario de atención de la biblioteca?":
        "La biblioteca abre a las 9hs.",

    "¿Está abierta durante toda la semana?":
        "La biblioteca abre todos los días de la semana.",

    "¿Puedo ir un domingo?":
        "La biblioteca abre todos los días de la semana.",

    "¿Qué días puedo acercarme a la biblioteca?":
        "La biblioteca abre todos los días de la semana.",


    # ==========================================
    # 2. PARAFRASEO - UBICACIÓN Y CONTACTO
    # ==========================================

    "Necesito saber dónde queda el establecimiento.":
        "La biblioteca está en la calle 123.",

    "¿Me podrían indicar la dirección para llegar?":
        "La biblioteca está en la calle 123.",

    "¿Cómo puedo comunicarme telefónicamente con ustedes?":
        "El número de teléfono de la biblioteca es 123456789.",

    "¿A qué número puedo llamar si tengo una consulta?":
        "El número de teléfono de la biblioteca es 123456789.",

    "¿Existe alguna manera de contactarlos por escrito?":
        "Sí, podés escribir a biblioteca@ejemplo.com.",

    "¿Cuál es la dirección de correo para enviar consultas?":
        "Sí, podés escribir a biblioteca@ejemplo.com.",


    # ==========================================
    # 3. PARAFRASEO - COLECCIÓN DE LIBROS
    # ==========================================

    "¿Hay material literario de ficción futurista o espacial?":
        "Si, la biblioteca tiene libros de ciencia ficción.",

    "Estoy buscando obras relacionadas con viajes espaciales o futuros imaginarios.":
        "Si, la biblioteca tiene libros de ciencia ficción.",

    "¿Disponen de bibliografía sobre acontecimientos históricos?":
        "Si, la biblioteca tiene libros de historia.",

    "¿Puedo encontrar material para aprender a programar?":
        "Si, la biblioteca tiene libros de programación.",

    "¿Hay bibliografía relacionada con las reacciones y sustancias químicas?":
        "Si, la biblioteca tiene libros de química.",

    "¿Tienen material que explique las leyes de la naturaleza y la mecánica?":
        "Si, la biblioteca tiene libros de física.",

    "Necesito bibliografía para estudiar números, álgebra y cálculo.":
        "Si, la biblioteca tiene libros de matemáticas.",

    "¿Hay libros sobre seres vivos, genética o células?":
        "Si, la biblioteca tiene libros de biología.",

    "¿Puedo consultar libros sobre países, mapas y territorios?":
        "Si, la biblioteca tiene libros de geografía.",


    # ==========================================
    # 4. NEGACIÓN Y DISTINCIÓN SEMÁNTICA
    # ==========================================

    "¿Hay textos escritos en algún idioma extranjero, por ejemplo inglés?":
        "No, la biblioteca no tiene libros en inglés.",

    "¿Disponen de ejemplares para practicar la lengua francesa?":
        "No, la biblioteca no tiene libros en francés.",

    "¿Se puede estudiar alemán mediante los libros de la biblioteca?":
        "No, la biblioteca no tiene libros en alemán.",

    "¿Tienen literatura italiana en su colección?":
        "No, la biblioteca no tiene libros en italiano.",

    "¿Hay libros orientados al crecimiento personal y la motivación?":
        "No, la biblioteca no tiene libros de autoayuda.",

    "¿Cuentan con libros de inglés o francés?":
        "No, la biblioteca no tiene libros en inglés.",

    "Busco novelas de ciencia ficción, no libros de historia.":
        "Si, la biblioteca tiene libros de ciencia ficción.",

    "¿Tienen libros de matemática, pero no de física?":
        "Si, la biblioteca tiene libros de matemáticas.",


    # ==========================================
    # 5. PREGUNTAS INDIRECTAS
    # ==========================================

    "Si quisiera pasar un día estudiando allí, ¿sería posible?":
        "Sí, hay salas de lectura y de estudio silenciosas.",

    "¿El lugar cuenta con espacios donde no haya mucho ruido?":
        "Sí, hay salas de lectura y de estudio silenciosas.",

    "No tengo computadora, ¿puedo utilizar alguna en el establecimiento?":
        "Sí, hay computadoras disponibles por hasta 1 hora por persona.",

    "¿Hay acceso a internet para los visitantes?":
        "Sí, el wifi es gratuito y la clave está en el mostrador.",

    "¿Tengo que pagar para formar parte de la biblioteca?":
        "La inscripción es gratuita.",

    "¿Qué requisitos debo cumplir para poder retirar material?":
        "Presentá tu DNI en el mostrador y completá el formulario de inscripción.",

    "¿Es necesario pagar alguna cuota mensual para estar registrado?":
        "La inscripción es gratuita.",


    # ==========================================
    # 6. PRÉSTAMOS - PREGUNTAS COMPLEJAS
    # ==========================================

    "¿Cuál es el límite de ejemplares que puedo retirar simultáneamente?":
        "Podés llevar hasta 3 libros a la vez.",

    "Si quiero llevarme varios libros a casa, ¿cuántos me permiten?":
        "Podés llevar hasta 3 libros a la vez.",

    "¿Cuál es el plazo máximo antes de tener que devolver un ejemplar?":
        "El préstamo dura 15 días.",

    "¿Cuánto tiempo tengo antes de que venza el préstamo?":
        "El préstamo dura 15 días.",

    "¿Puedo quedarme con el libro un mes si lo renuevo?":
        "Sí, podés renovarlo una vez por 15 días más si nadie lo reservó.",

    "¿Qué sucede si otra persona quiere el libro que tengo prestado?":
        "Sí, podés renovarlo una vez por 15 días más si nadie lo reservó.",

    "¿Se permite extender la fecha de devolución de un ejemplar?":
        "Sí, podés renovarlo una vez por 15 días más si nadie lo reservó.",

    "¿Existe algún castigo por no devolver el material a tiempo?":
        "Se aplica una suspensión de 1 día por cada día de demora.",

    "¿Qué consecuencias tiene atrasarse en la entrega?":
        "Se aplica una suspensión de 1 día por cada día de demora.",

    "Si pierdo un ejemplar que me prestaron, ¿cómo soluciono el problema?":
        "Acercate al mostrador para reponerlo o abonar su valor.",

    "¿Qué alternativas tengo si ya no encuentro el libro que retiré?":
        "Acercate al mostrador para reponerlo o abonar su valor.",

    "¿Puedo asegurarme de conseguir un libro antes de ir?":
        "Sí, podés reservarlo en el mostrador o por teléfono.",

    "¿Cómo hago para que no se lleven un libro que necesito?":
        "Sí, podés reservarlo en el mostrador o por teléfono.",


    # ==========================================
    # 7. PREGUNTAS CON DISTINTA ESTRUCTURA
    # ==========================================

    "¿La biblioteca cobra por utilizar sus instalaciones?":
        "La inscripción es gratuita.",

    "¿Es gratuita la conexión inalámbrica?":
        "Sí, el wifi es gratuito y la clave está en el mostrador.",

    "¿Puedo estacionar mi vehículo cerca mientras estudio?":
        "Sí, hay un estacionamiento gratuito para visitantes.",

    "¿Los días festivos puedo ir a leer o estudiar?":
        "No, la biblioteca permanece cerrada los feriados.",

    "¿Se puede acceder a la biblioteca cuando es feriado nacional?":
        "No, la biblioteca permanece cerrada los feriados.",

    "¿Puedo utilizar las instalaciones sin llevar mis propios dispositivos?":
        "Sí, hay computadoras disponibles por hasta 1 hora por persona.",

    "¿Hay algún espacio destinado exclusivamente a la lectura?":
        "Sí, hay salas de lectura y de estudio silenciosas.",

    "¿Puedo acceder a libros sin necesidad de comprarlos?":
        "Podés llevar hasta 3 libros a la vez.",


    # ==========================================
    # 8. PREGUNTAS ADVERSARIALES / DIFÍCILES
    # ==========================================

    "Si la biblioteca cierra a las 20, ¿a qué hora puedo comenzar a ir?":
        "La biblioteca abre a las 9hs.",

    "¿Cuál es la hora límite para devolver los libros?":
        "El préstamo dura 15 días.",

    "¿Cuántos días puedo conservar un libro si nadie más lo necesita?":
        "Sí, podés renovarlo una vez por 15 días más si nadie lo reservó.",

    "¿Cuánto tiempo puedo usar una computadora y cuántos libros puedo retirar?":
        "Sí, hay computadoras disponibles por hasta 1 hora por persona.",

    "¿Cuánto cuesta la inscripción y cuánto tiempo dura el préstamo?":
        "La inscripción es gratuita.",

    "¿Qué número debo marcar para consultar sobre libros, préstamos o reservas?":
        "El número de teléfono de la biblioteca es 123456789.",

    "¿Puedo llevarme cuatro libros y devolverlos después de dos semanas?":
        "Podés llevar hasta 3 libros a la vez.",

    "¿Puedo renovar el préstamo si otra persona ya reservó el libro?":
        "Sí, podés renovarlo una vez por 15 días más si nadie lo reservó.",

    "¿Qué ocurre si me atraso tres días en la devolución?":
        "Se aplica una suspensión de 1 día por cada día de demora.",

    "¿Hay estacionamiento, conexión a internet y computadoras disponibles?":
        "Sí, hay un estacionamiento gratuito para visitantes.",


    # ==========================================
    # 9. VARIACIONES COLOQUIALES
    # ==========================================

    "Che, ¿hasta qué hora está abierta?":
        "La biblioteca cierra a las 20hs.",

    "¿A qué hora arrancan a atender?":
        "La biblioteca abre a las 9hs.",

    "¿Me puedo llevar un par de libros para casa?":
        "Podés llevar hasta 3 libros a la vez.",

    "¿Hay compus para usar ahí?":
        "Sí, hay computadoras disponibles por hasta 1 hora por persona.",

    "¿Tienen internet para los que van?":
        "Sí, el wifi es gratuito y la clave está en el mostrador.",

    "¿Me puedo hacer socio sin pagar?":
        "La inscripción es gratuita.",

    "¿Y si me cuelgo y devuelvo tarde el libro?":
        "Se aplica una suspensión de 1 día por cada día de demora.",

    "¿Cómo hago para quedarme más tiempo con un libro?":
        "Sí, podés renovarlo una vez por 15 días más si nadie lo reservó.",

    "¿Se puede apartar un libro para después?":
        "Sí, podés reservarlo en el mostrador o por teléfono.",

    "¿Dónde puedo hablar con alguien que me ayude?":
        "Te derivo con el personal del mostrador, llamá al 123456789.",


    # ==========================================
    # 10. FUERA DEL DOMINIO / INFORMACIÓN NO DISPONIBLE
    # ==========================================

    "¿Qué películas están proyectando esta semana?":
        "No tengo información sobre la cartelera de películas.",

    "¿Qué clima va a hacer mañana?":
        "No tengo información sobre el clima.",

    "¿Quién es el director de la biblioteca?":
        "No tengo información sobre el director de la biblioteca.",

    "¿Cuántos empleados trabajan allí?":
        "No tengo información sobre la cantidad de empleados.",

    "¿Cuál es el libro más leído por los usuarios?":
        "No tengo información sobre los libros más leídos.",

    "¿Tienen algún evento programado para mañana?":
        "No tengo información sobre los eventos programados.",

    "¿Qué libros nuevos ingresaron este mes?":
        "No tengo información sobre las nuevas adquisiciones.",

    "¿Puedo reservar una sala privada para una reunión?":
        "No tengo información sobre la reserva de salas privadas.",

    "¿Se pueden imprimir documentos desde las computadoras?":
        "No tengo información sobre el servicio de impresión.",

    "¿Puedo llevar mascotas a la biblioteca?":
        "No tengo información sobre el ingreso de mascotas."
}

#Fuente: https://chatgpt.com/share/6b0111cf-0288-83e9-ba0b-812b56721a9d . Tambien se agrego una respuesta comodín si no hay coincidencia

Este nuevo diccionario sirve para pasarle preguntas con variaciones a las planteadas inicialmente. A su vez, se añaden las respuestas como método de verificación. De esta forma podemos saber si la respuesta estuvo bien o mal.

##Releve o indique cuáles respondió correctamente y cuáles no.

In [ ]:
filas = []
for consulta, esperada in pregresp_test.items():                                #Itera sobre las preguntas-respuestas de prueba
    r_tfidf = responder(consulta, "tfidf")                                      #Responde usando TFIDF
    r_emb   = responder(consulta, "emb")                                        #Responde usando embeddings
    resp_ok = esperada
    #Lista de diccionarios con la información para cada columna
    filas.append({
        "Consulta": consulta,
        "Esperada": "(no responder)" if esperada is None else esperada,
        "TF-IDF": r_tfidf,
        "TF-IDF ok": "✔" if r_tfidf == resp_ok else "✘",
        "Embeddings": r_emb,
        "Emb ok": "✔" if r_emb == resp_ok else "✘",
    })

#Conversión de la lista de diccionarios a un DataFrame
df_nuevas = pd.DataFrame(filas)

#Control de como se muestra el dataframe (máximo 45 caracteres)
pd.set_option("display.max_colwidth", 45)
display(df_nuevas)


#Impresión de la cantidad de preguntas correctamente respondidas respecto al total
print(f"TF-IDF:     {(df_nuevas['TF-IDF ok']=='✔').sum()}/{len(df_nuevas)}")
print(f"Embeddings: {(df_nuevas['Emb ok']=='✔').sum()}/{len(df_nuevas)}")

### 5) Añade tus conclusiones de todo lo realizado (2 punto)

* Resalte e indique en cuáles respuestas falla o tiene problemas.
* Cuáles preguntas confunde.
* Compare los resultados de los chatbots.







In [ ]:
#Analiza el dataframe anterior  y clasifica.

def clasificar_respuesta(respuesta, esperada):

    if respuesta == "Perdón, no se entendio la pregunta":                           #Este caso donde no se supera el umbral, no hay acierto
        return "No acierto"

    elif respuesta == esperada:                                                     #Si la respuesta es igual a la esperada, acierto
        return "Acierto"

    else:
        return "Confusión"                                                          #Si la respuesta no es igual a la esperada, es una confusión

#Fuente: https://chatgpt.com/share/6b021509-892c-83e9-a476-9dfc82ad7ad3

In [ ]:
#Se aplica la clasificación al Dataframe creando 2 nuevas columnas para luego graficar.

df_nuevas["TF-IDF resultado"] = df_nuevas.apply(
    lambda fila: clasificar_respuesta(
        fila["TF-IDF"],
        fila["Esperada"]
    ),
    axis=1
)

df_nuevas["Embeddings resultado"] = df_nuevas.apply(
    lambda fila: clasificar_respuesta(
        fila["Embeddings"],
        fila["Esperada"]
    ),
    axis=1
)

#Fuente: https://chatgpt.com/share/6b021509-892c-83e9-a476-9dfc82ad7ad3

In [ ]:
categorias = ["Acierto", "No acierto", "Confusión"]


#Tabla comparativa que cuenta cuántas veces cada método obtuvo cadade resultado
resumen = pd.DataFrame({
    #CUenta los resultados de cada método
    "TF-IDF": df_nuevas["TF-IDF resultado"].value_counts(),
    "Embeddings": df_nuevas["Embeddings resultado"].value_counts()
}).reindex(categorias, fill_value=0)                                            #Fuerza a las filas a seguir el ordend e las categorías

print(resumen)

#Fuente: https://chatgpt.com/share/6b021509-892c-83e9-a476-9dfc82ad7ad3

In [ ]:
#Gráfico de barras
ax = resumen.plot(
    kind="bar",
    figsize=(10, 6),
    width=0.8
)

#Agregar el valor encima de cada barra
for container in ax.containers:
    ax.bar_label(container, padding=3, fontsize=10)

plt.title("Comparación de respuestas: TF-IDF vs Embeddings")
plt.xlabel("Clasificación de respuestas")
plt.ylabel("Cantidad de respuestas")
plt.xticks(rotation=0)
plt.legend(title="Modelo")
plt.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

#Fuente: https://chatgpt.com/share/6b021509-892c-83e9-a476-9dfc82ad7ad3

Análisis: las preguntas  que fueron acertadas, no acertadas o confundidas se reflejan en el Dataframe de la consigna anterior:
* Si acertó, la respuesta esperada coincide con la respuesta que da cada chatbot
* Si la respuesta no coincide y es distinta a "Perdón, no se entendio la pregunta" (que es la respuesta comodín cuando no se alcanzó el umbral), el chatbot se confundió
* Si la respuesta de cada chatbot coincide con la respuesta comodín, entonces no acertó.

A fines de comodidad, se vuelve a visualizar el DataFrame (incluyendo el análisis de la respuesta)

In [ ]:
df_nuevas

Luego, se utiliza un gráfico de barras para comparar ambos chatbots. En primer lugar:

*  TFIDF: acertó 21/89 preguntas de testeo. Y se confundió en 68/89. No hubo "no aciertos".

*  Embeddings: acertó 17/89. Confundió 38/89 y no acertó 34/89.


Si nos enfocamos en los aciertos, el modelo TFIDF tuvo un mejor rendimiento.

Es importante entender el rol de las preguntas-respuestas y del umbral.

Si para evaluar ambos chatbots se usaran las preguntas-respuestas iniciales, probablemente acertarían la gran mayoría (por no decir todas), ya que las preguntas de test serían idénticas a las de la base, así que sus vectores coincidirían. Por eso es una buena práctica agregar nuevas preguntas de test que desafìen a los chatbots.

Si queremos mejorar el rendimiento, podemos aumentar la cantidad de preguntas-respuestas iniciales. Esta base de conocimiento es un conjunto de vectores. Mientras más vectores tengamos más posibilidades hay de que nuevos vectores (de nuevas consultas) encuentren uno similar. Sin embargo, tener preguntas que resulten en vectores muy cercanos (dentro de la base de conocimiento), podría fomentar la confusión, ya que quizás 2 preguntas difieren ligeramente en cuanto a su valor vectorial, pero esperan respuestas distintas. Al aparecer una nueva consulta, su vector podría inclinarse hacia el vector de la base de conocimiento incoherente.


En cuanto al umbral, puede ser crucial si queremos, por ejemplo, obtener "no aciertos" (como sucede para TFIDF), pero fomentaría la aparición de confusiones, ya que (por ejemplo con un umbral de 0.2) si los vectores no lograron alcanzar ese valor tan bajo, significa que muchos vectores realmente distan bastante de una respuesta correcta. Es justamente por eso, que tampoco es que embeddings sea mas preciso, solo tiene un umbral mayor (el chatbot se "anima" menos a dar una respuesta y responde como si no hubiese acierto)

En conclusión, ninguno de los dos chatbots logra un buen rendimiento con la base actual. TF-IDF acierta más pero responde siempre, mientras que embeddings es más cauteloso pero acierta menos. Para mejorar, convendría ampliar la base con más variantes de cada pregunta y calibrar el umbral de cada método.